# Field Alignment Demonstration
### SET Resonance Substrate Model

This notebook demonstrates **field alignment dynamics** in the SET substrate model.

Objectives:
- initialize a Charge gradient
- observe Spin alignment toward ∇C
- observe Temperature‑induced destabilization
- visualize alignment over time
- compute alignment metrics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from simulations.core.substrate_solver import (
    Grid, SubstrateParams, SubstrateState, step
)
from docs.resonance_substrate_model.tools.visualization.field_plotter import (
    plot_spin, plot_charge, plot_temperature, plot_scalar_gradient
)
from docs.resonance_substrate_model.tools.visualization.triadic_space_mapper import (
    visualize_triadic_state
)
from docs.resonance_substrate_model.tools.visualization.envelope_viewer import (
    visualize_envelope
)

## Simulation Parameters
Define grid, time step, and substrate parameters.

In [ ]:
nx, ny = 128, 128
dx = dy = 1.0

grid = Grid(nx=nx, ny=ny, dx=dx, dy=dy)

params = SubstrateParams(
    D_S=1.0, D_C=1.0, D_T=1.0,
    gamma_S=0.05, gamma_C=0.05, gamma_T=0.05,
    lambda_SC=1.2,   # stronger alignment
    lambda_ST=0.8,   # stronger destabilization
    beta_CS=0.0,     # no spin-induced charge
    eta_S=0.1, eta_C=0.1,
    T0=0.0
)

dt = 0.01
steps = 1500

## Initialize Charge Gradient
We create a **Gaussian Charge bump**, which produces a strong ∇C.

Spin starts random, so alignment can be observed clearly.

In [ ]:
state = SubstrateState(grid)

x = np.linspace(-1, 1, nx)
y = np.linspace(-1, 1, ny)
X, Y = np.meshgrid(x, y, indexing="ij")

# Gaussian Charge bump
sigma = 0.25
state.C = np.exp(-((X**2 + Y**2) / (2 * sigma**2)))

# Random Spin field
rng = np.random.default_rng(42)
state.S = rng.normal(scale=0.5, size=(nx, ny, 3))

# Uniform Temperature
state.T[:, :] = 0.0

plot_charge(state.C, title="Initial Charge Field")
plot_scalar_gradient(state.C, title="Initial |∇C|")
plot_spin(state.S, title="Initial Spin Field (Random)")

## Run Simulation
Store snapshots for alignment visualization.

In [ ]:
snapshots = []
snapshot_interval = 200

for i in range(steps):
    step(state, params, dt)
    if i % snapshot_interval == 0:
        snapshots.append((i, state.S.copy(), state.C.copy(), state.T.copy()))
        print(f"Step {i}/{steps}")

## Spin Alignment Over Time
Observe how Spin aligns with ∇C.

In [ ]:
fig, axes = plt.subplots(1, len(snapshots), figsize=(4*len(snapshots), 4))

for ax, (i, S, _, _) in zip(axes, snapshots):
    Sxy = S[..., :2]
    nx, ny = Sxy.shape[:2]
    Xg, Yg = np.meshgrid(np.arange(nx), np.arange(ny), indexing="ij")
    ax.quiver(Xg[::8,::8], Yg[::8,::8], Sxy[::8,::8,0], Sxy[::8,::8,1], scale=50)
    ax.set_title(f"Step {i}")
    ax.set_aspect('equal')

plt.show()

## Triadic Overlay (Final State)

In [ ]:
visualize_triadic_state(state, grid, prefix="Final Alignment State")

## Resonance Envelope Visualization
Shows where alignment is strong enough to form a resonance envelope.

In [ ]:
visualize_envelope(state.S, state.C, state.T, alpha=1.0, theta=0.1, dx=grid.dx, dy=grid.dy)

## Alignment Metric
Compute the dot product between Spin direction and ∇C direction.

In [ ]:
def alignment_metric(S, C, grid):
    grad = np.gradient(C, grid.dx, grid.dy)
    G = np.stack([grad[0], grad[1], np.zeros_like(grad[0])], axis=-1)

    S_norm = S / (np.linalg.norm(S, axis=-1, keepdims=True) + 1e-9)
    G_norm = G / (np.linalg.norm(G, axis=-1, keepdims=True) + 1e-9)

    return np.mean(np.sum(S_norm * G_norm, axis=-1))

alignment_values = []

state3 = SubstrateState(grid)
state3.C = np.exp(-((X**2 + Y**2) / (2 * sigma**2)))
state3.S = rng.normal(scale=0.5, size=(nx, ny, 3))

for i in range(steps):
    step(state3, params, dt)
    alignment_values.append(alignment_metric(state3.S, state3.C, grid))

plt.plot(alignment_values)
plt.title("Spin–Charge Alignment vs Time")
plt.xlabel("Step")
plt.ylabel("Alignment Metric")
plt.show()